# 텍스트 청킹 전략 (Text Chunking Strategies)

**Skilljar Lesson L02 대응**

이 노트북에서 다루는 내용:
1. 크기 기반 청킹 (Size-based Chunking)
2. 구조 기반 청킹 (Structure-based Chunking)
3. 의미 기반 청킹 (Semantic-based Chunking)
4. 문장 기반 청킹 (Sentence-based Chunking)
5. 청킹 전략 비교

In [ ]:
# ── Setup ──────────────────────────────────────────────
import re
import textwrap

## 샘플 문서

청킹 전략을 테스트할 샘플 텍스트를 준비합니다.  
건축공학 구조기준 스타일의 문서입니다.

In [ ]:
SAMPLE_DOCUMENT = """
# KDS 41 구조기준 (샘플)

## 1장 총칙

### 1.1 목적
이 기준은 건축물의 구조안전성을 확보하기 위한 최소한의 요구사항을 정한다. 건축구조기준은 건축물의 구조설계, 시공 및 유지관리에 적용된다. 모든 건축물은 자중, 적재하중, 풍하중, 지진하중 등에 안전하도록 설계되어야 한다.

### 1.2 적용 범위
이 기준은 신축, 증축, 개축 및 대수선하는 건축물에 적용한다. 기존 건축물의 구조안전성 평가에도 참고할 수 있다. 특수구조물에 대해서는 별도의 기준을 적용할 수 있다.

## 2장 하중

### 2.1 고정하중
고정하중은 구조물 자체의 무게와 영구적으로 부착된 부분의 무게를 포함한다. 콘크리트의 단위중량은 24 kN/m3, 철근콘크리트는 25 kN/m3을 표준값으로 한다. 벽돌벽의 단위중량은 18 kN/m3으로 한다.

### 2.2 적재하중
적재하중은 건축물의 용도에 따라 다르게 적용한다. 주거용 건물의 바닥 적재하중은 2.0 kN/m2, 사무실은 2.5 kN/m2, 상점은 4.0 kN/m2를 적용한다. 특수용도의 경우 실제 하중을 산정하여 적용해야 한다.

## 3장 콘크리트 구조

### 3.1 재료
콘크리트의 설계기준강도 fck는 최소 21 MPa 이상이어야 한다. 고강도 콘크리트의 경우 fck 40 MPa 이상을 적용할 수 있다. 철근의 항복강도 fy는 400 MPa 또는 500 MPa를 표준으로 한다.

### 3.2 보 설계
RC 보의 최소 철근비는 0.25*sqrt(fck)/fy 이상이어야 하며, 1.4/fy 이상이어야 한다. 최대 철근비는 균형 철근비의 0.75배를 초과할 수 없다. 전단보강은 스터럽 간격이 d/2 이하가 되도록 배치해야 한다.

### 3.3 기둥 설계
기둥의 최소 단면치수는 300mm 이상이어야 한다. 주근의 최소 개수는 4개이며, 최소 철근비는 0.01 이상이어야 한다. 띠철근의 간격은 주근 직경의 16배, 띠철근 직경의 48배, 기둥 최소 치수 중 가장 작은 값 이하로 한다.
""".strip()

print(f"문서 길이: {len(SAMPLE_DOCUMENT)}자")
print(f"줄 수: {SAMPLE_DOCUMENT.count(chr(10)) + 1}줄")

## §1. 크기 기반 청킹 (Size-based Chunking)

가장 기본적인 방법.  
고정된 문자 수로 텍스트를 분할하되, **오버랩**을 두어 문맥 단절을 최소화합니다.

In [ ]:
def chunk_by_char(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """텍스트를 고정 문자 수로 분할한다.

    Args:
        text: 분할할 텍스트
        chunk_size: 각 청크의 최대 문자 수
        overlap: 청크 간 겹치는 문자 수

    Returns:
        청크 리스트
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # 오버랩만큼 뒤로
    return chunks

In [ ]:
# 크기 기반 청킹 테스트
chunks_char = chunk_by_char(SAMPLE_DOCUMENT, chunk_size=300, overlap=50)

print(f"총 {len(chunks_char)}개 청크 생성\n")
for i, chunk in enumerate(chunks_char):
    print(f"--- 청크 {i+1} ({len(chunk)}자) ---")
    print(chunk[:100] + "...")
    print()

## §2. 구조 기반 청킹 (Structure-based Chunking)

문서의 **구조적 경계** (제목, 섹션)를 기준으로 분할합니다.  
Markdown 헤더 (`##`)를 경계로 사용하면 논리적 단위를 보존할 수 있습니다.

In [ ]:
def chunk_by_section(text: str, separator: str = "\n## ") -> list[str]:
    """텍스트를 섹션 구분자로 분할한다.

    Args:
        text: 분할할 텍스트 (Markdown 형식)
        separator: 섹션 구분자 (기본: Markdown H2)

    Returns:
        섹션별 청크 리스트
    """
    sections = text.split(separator)
    chunks = []
    for i, section in enumerate(sections):
        if section.strip():
            if i > 0:
                section = "## " + section
            chunks.append(section.strip())
    return chunks

In [ ]:
# 구조 기반 청킹 테스트
chunks_section = chunk_by_section(SAMPLE_DOCUMENT)

print(f"총 {len(chunks_section)}개 청크 생성\n")
for i, chunk in enumerate(chunks_section):
    first_line = chunk.split('\n')[0]
    print(f"--- 청크 {i+1} ({len(chunk)}자) ---")
    print(f"제목: {first_line}")
    print(f"미리보기: {chunk[:120]}...")
    print()

## §3. 문장 기반 청킹 (Sentence-based Chunking)

문장 단위로 분할하되, 일정 수의 문장을 묶어 하나의 청크로 만듭니다.

In [ ]:
def chunk_by_sentence(
    text: str,
    sentences_per_chunk: int = 5,
    overlap_sentences: int = 1
) -> list[str]:
    """문장 단위로 청킹한다.

    Args:
        text: 분할할 텍스트
        sentences_per_chunk: 청크당 문장 수
        overlap_sentences: 청크 간 겹치는 문장 수

    Returns:
        문장 그룹 청크 리스트
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s for s in sentences if s.strip()]

    chunks = []
    start = 0
    while start < len(sentences):
        end = min(start + sentences_per_chunk, len(sentences))
        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)
        start += sentences_per_chunk - overlap_sentences

    return chunks

In [ ]:
# 문장 기반 청킹 테스트
# Markdown 헤더를 제거한 본문만 사용
plain_text = re.sub(r'#+\s+.*\n', '', SAMPLE_DOCUMENT)
chunks_sentence = chunk_by_sentence(plain_text, sentences_per_chunk=3, overlap_sentences=1)

print(f"총 {len(chunks_sentence)}개 청크 생성\n")
for i, chunk in enumerate(chunks_sentence[:5]):
    print(f"--- 청크 {i+1} ({len(chunk)}자) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

## §4. 청킹 전략 비교

같은 문서에 대해 각 전략의 결과를 비교합니다.

In [ ]:
# 전략별 비교
strategies = {
    "크기 기반 (300자)": chunk_by_char(SAMPLE_DOCUMENT, 300, 50),
    "구조 기반 (##)": chunk_by_section(SAMPLE_DOCUMENT),
    "문장 기반 (3문장)": chunk_by_sentence(plain_text, 3, 1),
}

print(f"{'전략':<20} {'청크 수':>8} {'평균 크기':>10} {'최소':>8} {'최대':>8}")
print("-" * 60)
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    print(f"{name:<20} {len(chunks):>8} {sum(sizes)/len(sizes):>10.0f} {min(sizes):>8} {max(sizes):>8}")

## 정리

| 전략 | 장점 | 단점 | 적합한 경우 |
|------|------|------|------------|
| 크기 기반 | 구현 간단, 균일한 크기 | 문맥 단절 가능 | 균일한 텍스트 |
| 구조 기반 | 논리적 단위 유지 | 크기 불균일 | 구조화된 문서 |
| 문장 기반 | 문법적 완결성 | 짧은 청크 발생 | 대화형 텍스트 |

다음 노트북에서는 이 청크들을 **임베딩 벡터**로 변환합니다. → `S4_02_embeddings.ipynb`